# Vehicle Detection → Damage Classification → Severity Pipeline

This notebook joins your **3 models** into one video pipeline:

1. **Vehicle detector** — `car-75e-11n.pt` (YOLO) — finds vehicles in each video frame.
2. **Damage classifier** — `model_mobilenetv2.keras` (Keras) — for each detected vehicle crop, decides **damaged / not damaged**.
3. **Severity model** — `car-damage.pt` (YOLO) — only runs on crops classified as *damaged*, and outputs **minor / moderate / severe**.

**Flow per frame:**
```
video frame ──▶ [Model 1: detect vehicles] ──▶ crop each vehicle
                                                     │
                                                     ▼
                                    [Model 2: damaged / not damaged?]
                                                     │
                                        not damaged ─┴─ damaged
                                            │              │
                                          skip     [Model 3: minor/moderate/severe]
```

Upload your video and the 3 model files to `/content/` (or update the paths in the config cell), then run all cells top to bottom.


## 1. Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless tensorflow tqdm

## 2. Imports

In [ ]:
import os
import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from ultralytics import YOLO
from tensorflow.keras.models import load_model


## 3. Config — edit these paths / settings

- `VEHICLE_MODEL_PATH` — Model 1 (vehicle presence detector)
- `DAMAGE_CLASSIFIER_PATH` — Model 2 (damaged / not damaged, Keras)
- `SEVERITY_MODEL_PATH` — Model 3 (minor / moderate / severe, YOLO)
- `VIDEO_PATH` — the input video to process
- `OUTPUT_DIR` — where annotated video + results get saved

`FRAME_SAMPLE_RATE`: process every Nth frame (1 = every frame, slower but most thorough;
5-10 is usually enough for a smooth-moving dashcam/drone video and much faster).


In [ ]:
# ---- Model paths ----
VEHICLE_MODEL_PATH = "/content/car-75e-11n.pt"
DAMAGE_CLASSIFIER_PATH = "/content/drive/MyDrive/car-damage-models/model_mobilenetv2.keras"
SEVERITY_MODEL_PATH = "/content/car-damage (1).pt"

# ---- Video ----
VIDEO_PATH = "/content/Rubble and damaged cars in street after new quake hit Spains GranadaRubble and damaged cars can.mp4"
OUTPUT_DIR = "/content/output"

# ---- Inference settings ----
FRAME_SAMPLE_RATE = 5          # analyze every Nth frame
VEHICLE_CONF_THRES = 0.35      # confidence threshold for Model 1 (vehicle detector)
SEVERITY_CONF_THRES = 0.25     # confidence threshold for Model 3 (severity detector)
CLASSIFIER_IMG_SIZE = (224, 224)   # input size expected by model_mobilenetv2.keras
# If your classifier outputs a single sigmoid unit, index 0 = P(damaged).
# If it outputs 2 softmax units, set the label order below to match training.
DAMAGE_CLASS_NAMES = ["00-damaged", "01-whole"]   # index order must match training generator's class_indices

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Load the 3 models

In [ ]:
print("Loading Model 1 - vehicle detector ...")
vehicle_model = YOLO(VEHICLE_MODEL_PATH)

print("Loading Model 2 - damage classifier (Keras) ...")
damage_classifier = load_model(DAMAGE_CLASSIFIER_PATH)

print("Loading Model 3 - severity detector ...")
severity_model = YOLO(SEVERITY_MODEL_PATH)

print("All 3 models loaded.")
print("Vehicle detector classes:", vehicle_model.names)
print("Severity model classes:", severity_model.names)


## 5. Helper functions

- `crop_box` — safely crops a detection box out of a frame.
- `classify_damage` — runs Model 2 on a crop, returns `(label, confidence)`.
- `get_severity` — runs Model 3 on a crop, returns the top severity class + confidence
  (or `None` if Model 3 finds nothing above threshold on that crop).


In [ ]:
def crop_box(frame, xyxy, pad=0.05):
    """Crop a box out of a frame with a small padding margin, clipped to frame bounds."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = xyxy
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - bw * pad))
    y1 = max(0, int(y1 - bh * pad))
    x2 = min(w, int(x2 + bw * pad))
    y2 = min(h, int(y2 + bh * pad))
    return frame[y1:y2, x1:x2]


def preprocess_for_classifier(crop):
    """Resize + normalize a BGR crop for model_mobilenetv2.keras."""
    img = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, CLASSIFIER_IMG_SIZE)
    img = img.astype("float32") / 255.0
    return np.expand_dims(img, axis=0)


def classify_damage(crop):
    """Run Model 2 on a single vehicle crop. Returns (label:str, confidence:float)."""
    if crop.size == 0:
        return "unknown", 0.0
    x = preprocess_for_classifier(crop)
    pred = damage_classifier.predict(x, verbose=False)

    if pred.shape[-1] == 1:
        # single sigmoid output -> P(damaged)
        p_damaged = float(pred[0][0])
        label = DAMAGE_CLASS_NAMES[1] if p_damaged >= 0.5 else DAMAGE_CLASS_NAMES[0]
        conf = p_damaged if label == DAMAGE_CLASS_NAMES[1] else 1 - p_damaged
    else:
        # softmax over N classes
        idx = int(np.argmax(pred[0]))
        conf = float(pred[0][idx])
        label = DAMAGE_CLASS_NAMES[idx] if idx < len(DAMAGE_CLASS_NAMES) else str(idx)

    return label, conf


def get_severity(crop):
    """Run Model 3 on a damaged crop. Returns (severity_label, confidence) or (None, 0.0)."""
    if crop.size == 0:
        return None, 0.0
    results = severity_model.predict(crop, conf=SEVERITY_CONF_THRES, verbose=False)
    r = results[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None, 0.0
    # take the highest-confidence detection on this crop
    confs = r.boxes.conf.cpu().numpy()
    best_i = int(np.argmax(confs))
    cls_id = int(r.boxes.cls[best_i].item())
    label = severity_model.names[cls_id]
    conf = float(confs[best_i])
    return label, conf


## 6. Main pipeline — process the video

For each sampled frame:
1. Run Model 1 → get vehicle boxes.
2. For each vehicle box → crop it, run Model 2 (damaged / not damaged).
3. If damaged → run Model 3 on that same crop for severity.
4. Draw a box + label on the frame (color-coded), and log a row per detection.

Output: an annotated video (`annotated_output.mp4`) and a results table (`results.csv`).


In [ ]:
SEVERITY_COLORS = {
    "01-minor": (0, 255, 255),      # yellow (BGR)
    "02-moderate": (0, 165, 255),   # orange
    "03-severe": (0, 0, 255),       # red
}
NOT_DAMAGED_COLOR = (0, 200, 0)   # green


def process_video(video_path, output_dir, sample_rate=FRAME_SAMPLE_RATE):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = os.path.join(output_dir, "annotated_output.mp4")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps / max(sample_rate, 1), (width, height))

    rows = []
    frame_idx = 0
    pbar = tqdm(total=total_frames, desc="Processing video")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        pbar.update(1)

        if frame_idx % sample_rate != 0:
            frame_idx += 1
            continue

        # ---- Model 1: vehicle detection ----
        det_results = vehicle_model.predict(frame, conf=VEHICLE_CONF_THRES, verbose=False)
        boxes = det_results[0].boxes

        if boxes is not None and len(boxes) > 0:
            for b in boxes:
                xyxy = b.xyxy[0].cpu().numpy()
                vehicle_conf = float(b.conf[0].item())
                crop = crop_box(frame, xyxy)

                # ---- Model 2: damaged or not ----
                damage_label, damage_conf = classify_damage(crop)
                is_damaged = damage_label == DAMAGE_CLASS_NAMES[1]

                severity_label, severity_conf = (None, 0.0)
                if is_damaged:
                    # ---- Model 3: severity, only for damaged vehicles ----
                    severity_label, severity_conf = get_severity(crop)

                rows.append({
                    "frame": frame_idx,
                    "timestamp_sec": round(frame_idx / fps, 2),
                    "vehicle_conf": round(vehicle_conf, 3),
                    "damage_label": damage_label,
                    "damage_conf": round(damage_conf, 3),
                    "severity_label": severity_label,
                    "severity_conf": round(severity_conf, 3) if severity_label else None,
                    "x1": int(xyxy[0]), "y1": int(xyxy[1]), "x2": int(xyxy[2]), "y2": int(xyxy[3]),
                })

                # ---- draw annotation ----
                x1, y1, x2, y2 = map(int, xyxy)
                if is_damaged and severity_label:
                    color = SEVERITY_COLORS.get(severity_label.lower(), (0, 0, 255))
                    text = f"damaged: {severity_label} ({severity_conf:.2f})"
                elif is_damaged:
                    color = (0, 0, 255)
                    text = f"damaged ({damage_conf:.2f})"
                else:
                    color = NOT_DAMAGED_COLOR
                    text = f"ok ({damage_conf:.2f})"

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, text, (x1, max(20, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        writer.write(frame)
        frame_idx += 1

    pbar.close()
    cap.release()
    writer.release()

    df = pd.DataFrame(rows)
    csv_path = os.path.join(output_dir, "results.csv")
    df.to_csv(csv_path, index=False)

    return out_path, csv_path, df


In [ ]:
annotated_video_path, csv_path, results_df = process_video(VIDEO_PATH, OUTPUT_DIR)
print("Annotated video saved to:", annotated_video_path)
print("Results table saved to:", csv_path)
results_df.head(200)


## 7. Summary report

In [ ]:
if len(results_df) == 0:
    print("No vehicles detected in the sampled frames.")
else:
    total_detections = len(results_df)
    damaged_df = results_df[results_df["damage_label"] == DAMAGE_CLASS_NAMES[1]]
    not_damaged_count = total_detections - len(damaged_df)

    print(f"Total vehicle detections (sampled frames): {total_detections}")
    print(f"Not damaged: {not_damaged_count}")
    print(f"Damaged: {len(damaged_df)}")

    if len(damaged_df) > 0:
        print("\nSeverity breakdown (damaged vehicles only):")
        print(damaged_df["severity_label"].value_counts(dropna=False))


## 8. (Optional) Download outputs

Run this cell in Colab to download the annotated video and CSV report to your machine.


In [ ]:
from google.colab import files
files.download(annotated_video_path)
files.download(csv_path)


---
### Notes / things to double check before running

- **Model 2 output format**: the code auto-handles either a single sigmoid unit or a 2-class
  softmax output. Just make sure `DAMAGE_CLASS_NAMES = ["not_damaged", "damaged"]` matches the
  class order your classifier was trained with (check `train_generator.class_indices` if you
  still have that from training).
- **Model 3 (`car-damage.pt`) is only run on crops Model 2 flagged as damaged** — this saves
  compute and matches the workflow you described (detect → classify damaged/not → severity).
- **`FRAME_SAMPLE_RATE`**: raise this (e.g. 10-15) for long videos to speed things up; lower it
  (e.g. 1-2) if vehicles move fast and you don't want to miss detections.
- If your video is long, consider running this on a Colab GPU runtime (`Runtime > Change runtime type > GPU`)
  — both YOLO models will run much faster.


In [ ]:
!pip install -q ultralytics opencv-python-headless tensorflow matplotlib

In [ ]:


## 0. Setup — install packages & set paths
!pip install -q ultralytics opencv-python-headless tensorflow matplotlib
import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from ultralytics import YOLO
from tensorflow.keras.models import load_model

# ---- Model paths ----
VEHICLE_MODEL_PATH = "/content/car-75e-11n.pt"
DAMAGE_CLASSIFIER_PATH = "/content/drive/MyDrive/car-damage-models/model_mobilenetv2.keras"
SEVERITY_MODEL_PATH = "/content/car-damage (1).pt"

# ---- Video ----
VIDEO_PATH = "/content/Rubble and damaged cars in street after new quake hit Spains GranadaRubble and damaged cars can.mp4"

# ---- Working folders (each stage reads/writes its own folder) ----
DETECTED_VEHICLES_DIR = "/content/1_detected_vehicles"
ANNOTATED_FRAMES_DIR = "/content/1_annotated_frames"
DAMAGED_VEHICLES_DIR = "/content/2_damaged_vehicles"
SEVERITY_RESULTS_DIR = "/content/3_severity_results"

for d in [DETECTED_VEHICLES_DIR, ANNOTATED_FRAMES_DIR, DAMAGED_VEHICLES_DIR, SEVERITY_RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

FRAME_SAMPLE_RATE = 5     # process every Nth frame of the video
VEHICLE_CONF_THRES = 0.35
SEVERITY_CONF_THRES = 0.25


def show_images(image_paths, titles=None, cols=4, figsize_per=(3.2, 3.2)):
    """Small helper: show a grid of images with matplotlib."""
    n = len(image_paths)
    if n == 0:
        print("Nothing to show.")
        return
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per[0]*cols, figsize_per[1]*rows))
    axes = np.array(axes).reshape(-1)
    for i, p in enumerate(image_paths):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        axes[i].axis("off")
        if titles is not None:
            axes[i].set_title(titles[i], fontsize=9)
    for j in range(n, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

## Part 1 — Vehicle detection (`car-75e-11n.pt`)


print("Loading Model 1 - vehicle detector ...")
vehicle_model = YOLO(VEHICLE_MODEL_PATH)
print("Classes:", vehicle_model.names)

def crop_box(frame, xyxy, pad=0.05):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = xyxy
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - bw * pad))
    y1 = max(0, int(y1 - bh * pad))
    x2 = min(w, int(x2 + bw * pad))
    y2 = min(h, int(y2 + bh * pad))
    return frame[y1:y2, x1:x2]


cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

frame_idx = 0
detected_crop_paths = []
annotated_frame_paths = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SAMPLE_RATE == 0:
        results = vehicle_model.predict(frame, conf=VEHICLE_CONF_THRES, verbose=False)
        boxes = results[0].boxes

        if boxes is not None and len(boxes) > 0:
            for veh_idx, b in enumerate(boxes):
                xyxy = b.xyxy[0].cpu().numpy()
                conf = float(b.conf[0].item())
                crop = crop_box(frame, xyxy)
                if crop.size == 0:
                    continue
                crop_name = f"frame{frame_idx:05d}_veh{veh_idx}.jpg"
                crop_path = os.path.join(DETECTED_VEHICLES_DIR, crop_name)
                cv2.imwrite(crop_path, crop)
                detected_crop_paths.append(crop_path)

            # save an annotated version of the full frame too
            annotated = results[0].plot()  # ultralytics draws boxes+labels for us
            ann_path = os.path.join(ANNOTATED_FRAMES_DIR, f"frame{frame_idx:05d}.jpg")
            cv2.imwrite(ann_path, annotated)
            annotated_frame_paths.append(ann_path)

    frame_idx += 1

cap.release()
print(f"Sampled frames processed: {len(annotated_frame_paths)}")
print(f"Vehicle crops saved: {len(detected_crop_paths)}  -> {DETECTED_VEHICLES_DIR}")

# Show a sample of annotated frames (full frame with detection boxes drawn)
print("Annotated frames (vehicle detection boxes):")
show_images(annotated_frame_paths[:8], titles=[Path(p).name for p in annotated_frame_paths[:8]])

# Show a sample of the cropped detected-vehicle images themselves
print("Cropped detected-vehicle images:")
show_images(detected_crop_paths[:12], titles=[Path(p).name for p in detected_crop_paths[:12]])


## Part 2 — Damage classification (`model_mobilenetv2.keras`)


print("Loading Model 2 - damage classifier (Keras) ...")
damage_classifier = load_model(DAMAGE_CLASSIFIER_PATH)

CLASSIFIER_IMG_SIZE = (224, 224)
# index order must match the classifier's training class_indices
DAMAGE_CLASS_NAMES = ["not_damaged", "damaged"]

def preprocess_for_classifier(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, CLASSIFIER_IMG_SIZE)
    img = img.astype("float32") / 255.0
    return np.expand_dims(img, axis=0)


def classify_damage(img_bgr):
    x = preprocess_for_classifier(img_bgr)
    pred = damage_classifier.predict(x, verbose=False)
    if pred.shape[-1] == 1:
        p_damaged = float(pred[0][0])
        label = DAMAGE_CLASS_NAMES[1] if p_damaged >= 0.5 else DAMAGE_CLASS_NAMES[0]
        conf = p_damaged if label == DAMAGE_CLASS_NAMES[1] else 1 - p_damaged
    else:
        idx = int(np.argmax(pred[0]))
        conf = float(pred[0][idx])
        label = DAMAGE_CLASS_NAMES[idx] if idx < len(DAMAGE_CLASS_NAMES) else str(idx)
    return label, conf


classification_results = []   # (path, label, conf)
damaged_crop_paths = []

for crop_path in detected_crop_paths:
    img = cv2.imread(crop_path)
    if img is None:
        continue
    label, conf = classify_damage(img)
    classification_results.append((crop_path, label, conf))

    if label == DAMAGE_CLASS_NAMES[1]:
        dst = os.path.join(DAMAGED_VEHICLES_DIR, Path(crop_path).name)
        shutil.copy(crop_path, dst)
        damaged_crop_paths.append(dst)

print(f"Classified: {len(classification_results)} images")
print(f"Predicted damaged: {len(damaged_crop_paths)}  -> {DAMAGED_VEHICLES_DIR}")

# Show classification results: each crop with its predicted label
sample = classification_results[:12]
show_images(
    [p for p, _, _ in sample],
    titles=[f"{label} ({conf:.2f})" for _, label, conf in sample],
)

# Show only the ones predicted as damaged (going on to Part 3)
print("Vehicles classified as DAMAGED:")
show_images(damaged_crop_paths[:12], titles=[Path(p).name for p in damaged_crop_paths[:12]])


## Part 3 — Severity detection (`car-damage.pt`)


print("Loading Model 3 - severity detector ...")
severity_model = YOLO(SEVERITY_MODEL_PATH)
print("Classes:", severity_model.names)

severity_result_paths = []
severity_summary = []   # (filename, severity_label, conf) or (filename, None, 0.0)

for damaged_path in damaged_crop_paths:
    img = cv2.imread(damaged_path)
    results = severity_model.predict(img, conf=SEVERITY_CONF_THRES, verbose=False)
    r = results[0]

    annotated = r.plot()  # draws severity boxes + labels on the crop
    out_path = os.path.join(SEVERITY_RESULTS_DIR, Path(damaged_path).name)
    cv2.imwrite(out_path, annotated)
    severity_result_paths.append(out_path)

    if r.boxes is not None and len(r.boxes) > 0:
        confs = r.boxes.conf.cpu().numpy()
        best_i = int(np.argmax(confs))
        cls_id = int(r.boxes.cls[best_i].item())

        severity_summary.append((Path(damaged_path).name, severity_model.names[cls_id], float(confs[best_i])))
    else:
        severity_summary.append((Path(damaged_path).name, None, 0.0))

print(f"Severity results saved -> {SEVERITY_RESULTS_DIR}")

# Show final severity results: each damaged vehicle with its severity box/label
show_images(
    severity_result_paths[:12],
    titles=[f"{name}: {sev if sev else 'no detection'}" for name, sev, conf in severity_summary[:12]],
)

# Text summary of severity per damaged vehicle
for name, sev, conf in severity_summary:
    if sev:
        print(f"{name}: {sev} ({conf:.2f})")
    else:
        print(f"{name}: severity model found nothing above threshold")

